# 📊 CVM Data Extraction - Elite Tickers B3

Este notebook implementa a extração otimizada de dados financeiros (DRE) diretamente do portal de dados abertos da CVM, focando estritamente em uma lista de elite de tickers da B3.

**Destaques:**
1. **Mapeamento Preciso:** Converte Tickers em Códigos CVM via arquivos FCA.
2. **Big Data Filter:** Filtra dados brutos de DRE desde 2015.
3. **Memória Otimizada:** Descarta empresas irrelevantes logo na leitura dos arquivos ZIP.

In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os
import time
from datetime import datetime
from pathlib import Path

# Configurações de pastagem
PASTA_OUTPUT = Path("../data/raw")
PASTA_OUTPUT.mkdir(parents=True, exist_ok=True)

print("✅ Ambiente configurado.")

## 1. Definição do Universo Alvo
Lista de tickers de elite que desejamos monitorar.

In [ ]:
TICKERS_ALVO = [
    "PETR4","ITUB4","VALE3","BPAC11","ABEV3","WEGE3","BBDC3","AXIA3","ITSA4","BBAS3","VIVT3","SANB11","SBSP3",
    "B3SA3","RDOR3","SUZB3","BBSE3","EMBJ3","CPLE3","TIMS3","RENT3","CPFE3","CXSE3","EQTL3","PRIO3","RADL3",
    "NEOE3","ENEV3","GGBR3","EGIE3","CMIG4","VBBR3","PSSA3","MOTV3","CMIN3","RAIL3","UGPA3","MBRF3","ENGI11",
    "KLBN11","CSAN3","TOTS3","CSMG3","ISAE4","CGAS3","MULT3","BPAN4","REDE3","ALOS3","MRSA3B","LREN3","TAEE11",
    "EQPA3","CEGR3","HYPE3","GGPS3","SAPR11","CYRE3","CEEB3","GMAT3","AURE3","BNBR3","AZUL54","SMFT3","NATU3",
    "ALUP11","ASAI3","GOAU4","ENMT4","CURY3","CSNA3","FLRY3","ALPA4","BRAP4","USIM5","BRAV3","TTEN3","EKTR3",
    "CASN3","SLCE3","POMO4","BMEB4","IGTI3","DIRR3","SRNA3","MDIA3","BRSR6","UNIP6","ODPV3","VIVA3","BRKM5",
    "RAIZ4","MGLU3","ECOR3","ORVR3","COGN3","FRAS3","WHRL4","CBAV3","ABCB4","JHSF3","SMTO3","VULC3","HBSA3",
    "EQMA3B","CLSC4","MRVE3","SHUL4","AZZA3","HAPV3","SIMH3","BAZA3","BEEF3","IRBR3","LEVE3","MOVI3","DXCO3",
    "RIAA3","DASA3","CBEE3","VAMO3","GRND3","INTB3","PGMN3","EZTC3","CEAB3","RECV3","TEND3","MILS3","LAVV3",
    "LOGN3","COCE5","YDUQ3","FESA4","GEPA4","EMAE4","MDNE3","BEES3","BMGB4","PINE4","PLPL3","SBFG3","BHIA3",
    "AUAU3","TGMA3","TFCO4","ONCO3","LOGG3","BLAU3","CSED3","CAML3","PNVL3","RAPT4","RANI3","JSLG3","CEBR3",
    "BSLI3","AGRO3","FIQE3","BMOB3","ARML3","MATD3","LWSA3","EUCA3","VTRU3","OPCT3","ANIM3","LIGT3","TRIS3",
    "VLID3","TUPY3","SEQL3","EVEN3","DESK3","MYPK3","KEPL3","WIZC3","SEER3","ZAMP3","OFSA3","BRST3","TELB3",
    "PCAR3","OBTC3","PRNR3","SOJA3","GPAR3","PFRM3","BGIP4","CVCB3","ALPK3","FRIO3","MOSI3","MLAS3","AMER3",
    "DEXP3","JALL3","BIOM3","LAND3","WLMM3","SCAR3","CSUD3","TASA3","MELK3","PATI3","ROMI3","AALR3","SYNE3",
    "PEAB3","ALLD3","CEED3","CGRA4","TKNO4","MERC4","VITT3","MTSA4","POSI3","QUAL3","CRPG5","TECN3","AMOB3",
    "RNEW4","AFLT3","VVEO3","AMAR3","PTBL3","RPAD3","VSTE3","DOHL4","LJQQ3","ESPA3","MTRE3","PTNT4","HBOR3",
    "CAMB3","CASH3","AMBP3","DMVF3","MEAL3","EALT4","HBRE3","MNP3"
]

## 2. Tradução Ticker -> Código CVM
Mapeamos os códigos numéricos da CVM necessários para baixar os balanços.

In [ ]:
def gerar_mapa_cvm_focado(tickers_alvo):
    ano_referencia = datetime.now().year - 1
    print(f"🕵️‍♂️ Mapeando Códigos CVM (Ano Base: {ano_referencia})...")
    url_fca = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/FCA/DADOS/fca_cia_aberta_{ano_referencia}.zip"
    
    resposta = requests.get(url_fca, timeout=60)
    arquivo_zip = zipfile.ZipFile(io.BytesIO(resposta.content))
    nome_ficheiro_acoes = f"fca_cia_aberta_valor_mobiliario_{ano_referencia}.csv"
    
    dicionario_cvm_ticker = {}
    with arquivo_zip.open(nome_ficheiro_acoes) as f:
        df_fca = pd.read_csv(f, sep=';', encoding='ISO-8859-1')
        col_ticker = 'Codigo_Negociacao'
        col_cvm = 'Codigo_CVM'
        
        # Tratamento de nomes de colunas
        df_fca.columns = [c.upper() for c in df_fca.columns]
        col_ticker = 'CODIGO_NEGOCIACAO'
        col_cvm = 'CODIGO_CVM'
        
        df_fca = df_fca.dropna(subset=[col_ticker, col_cvm])
        for cd_cvm, grupo in df_fca.groupby(col_cvm):
            tickers = [str(t).strip() for t in grupo[col_ticker].unique()]
            for t in tickers:
                if t in tickers_alvo:
                    dicionario_cvm_ticker[int(cd_cvm)] = t
                    break
    return dicionario_cvm_ticker

mapa_cvm = gerar_mapa_cvm_focado(TICKERS_ALVO)
print(f"✅ Mapeados {len(mapa_cvm)} ativos.")

## 3. Extração dos Balanços (DRE Histórico)
Loop pelos anos solicitados capiturando Receita e Lucro.

In [ ]:
def extrair_balancos(mapa_cvm, ano_inicio=2015, ano_fim=2024):
    mapa_contas = {
        '3.01': '01_Receita_Liquida_R$',
        '3.03': '02_Lucro_Bruto_R$',
        '3.05': '03_EBIT_Operacional_R$',
        '3.06': '04_Resultado_Financeiro_R$',
        '3.09': '05_Lucro_Antes_Impostos_EBT_R$',
        '3.11': '06_Lucro_Liquido_R$'
    }
    contas_alvo = list(mapa_contas.keys())
    codigos_cvm_alvo = list(mapa_cvm.keys())
    lista_balancos = []

    for ano in range(ano_inicio, ano_fim + 1):
        print(f"🚀 Processando {ano}...")
        url = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
        try:
            res = requests.get(url, timeout=30)
            if res.status_code == 200:
                zp = zipfile.ZipFile(io.BytesIO(res.content))
                fname = f"dfp_cia_aberta_DRE_con_{ano}.csv"
                with zp.open(fname) as f:
                    df = pd.read_csv(f, sep=';', encoding='ISO-8859-1')
                    df = df[df['CD_CVM'].isin(codigos_cvm_alvo)]
                    df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']
                    df_f = df[df['CD_CONTA'].isin(contas_alvo)].copy()
                    df_f['Conta_Nome'] = df_f['CD_CONTA'].map(mapa_contas)
                    lista_balancos.append(df_f)
        except: pass
            
    return pd.concat(lista_balancos, ignore_index=True) if lista_balancos else pd.DataFrame()

df_raw = extrair_balancos(mapa_cvm)
print(f"✅ {len(df_raw)} registros capturados.")

## 4. Pivotagem e Exportação
Transformamos as linhas em colunas e salvamos o CSV final.

In [ ]:
if not df_raw.empty:
    df_pivot = df_raw.pivot_table(
        index=['DENOM_CIA', 'CD_CVM', 'DT_FIM_EXERC'],
        columns='Conta_Nome',
        values='VL_CONTA',
        aggfunc='sum'
    ).reset_index()
    
    # Ajuste de escala e datas
    df_pivot['Ticker_Alvo'] = df_pivot['CD_CVM'].map(mapa_cvm)
    df_pivot['Ano_Exercicio'] = pd.to_datetime(df_pivot['DT_FIM_EXERC']).dt.year
    
    # Restaurar unidades monetárias (Milhares -> Reais)
    cols_dinheiro = [c for c in df_pivot.columns if 'R$' in c]
    for col in cols_dinheiro: df_pivot[col] = df_pivot[col] * 1000
    
    df_final = df_pivot.sort_values(['Ticker_Alvo', 'Ano_Exercicio'])
    caminho = PASTA_OUTPUT / "05_CVM_Historico_Focado.csv"
    df_final.to_csv(caminho, sep=';', decimal=',', index=False, encoding='utf-8-sig')
    print(f"📁 Arquivo salvo em: {caminho}")
    display(df_final.head(10))